In [1]:
# Importaciones necesarias para el servidor web y manejo de datos espaciales
from flask import Flask, request, jsonify
from arcgis.gis import GIS
import json
import os
import time
import threading

# Inicializacion de la aplicacion Flask y variables globales
app = Flask(__name__)
GEOJSON_FILE = "gps_data.geojson"
file_lock = threading.Lock()

# Conexion al motor de ArcGIS
print("Conectando al motor de ArcGIS...")
gis_conexion = GIS()
print("Conexion establecida para Flask con exito.")

# Funciones para manejar el archivo local de coordenadas
def init_geojson():
    with open(GEOJSON_FILE, "w") as f:
        json.dump({"type": "FeatureCollection", "features": []}, f)

def load_geojson():
    try:
        with open(GEOJSON_FILE, "r") as f:
            content = f.read().strip()
            if not content:
                raise ValueError("Archivo vacio")
            return json.loads(content)
    except (json.JSONDecodeError, ValueError, FileNotFoundError):
        init_geojson()
        return {"type": "FeatureCollection", "features": []}

if not os.path.exists(GEOJSON_FILE):
    init_geojson()

# Ruta principal que sirve la interfaz web del tracker
@app.route("/")
def index():
    return """
<!DOCTYPE html>
<html lang="es">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>GPS Tracker</title>
    <style>
        body { font-family: sans-serif; max-width: 400px; margin: 40px auto; padding: 20px; }
        button { padding: 14px 28px; font-size: 16px; cursor: pointer; border-radius: 8px;
                 border: none; background: #2563eb; color: white; width: 100%; }
        button:disabled { background: #93c5fd; cursor: not-allowed; }
        #estado { margin-top: 20px; padding: 14px; border-radius: 8px;
                  background: #f1f5f9; font-size: 14px; white-space: pre-line; }
        .ok    { background: #dcfce7 !important; color: #166534; }
        .error { background: #fee2e2 !important; color: #991b1b; }
    </style>
</head>
<body>
    <h2>GPS Tracker + ArcGIS</h2>
    <button id="btn" onclick="iniciar()">Iniciar captura</button>
    <div id="estado">Esperando...</div>

<script>
    let intervalo = null;
    let activo = false;

    function log(msg, tipo) {
        const el = document.getElementById("estado");
        el.textContent = msg;
        el.className = tipo || "";
    }

    function iniciar() {
        if (!navigator.geolocation) {
            log("Geolocalizacion no soportada.", "error");
            return;
        }
        if (activo) {
            clearInterval(intervalo);
            activo = false;
            document.getElementById("btn").textContent = "Iniciar captura";
            log("Captura detenida.");
            return;
        }
        activo = true;
        document.getElementById("btn").textContent = "Detener captura";
        log("Obteniendo ubicacion...");
        capturar();
        intervalo = setInterval(capturar, 2000);
    }

    function capturar() {
        navigator.geolocation.getCurrentPosition(
            async function(pos) {
                const lat = pos.coords.latitude;
                const lon = pos.coords.longitude;
                const acc = pos.coords.accuracy;

                log(
                    `Ubicacion enviada\\n` +
                    `Lat: ${lat.toFixed(6)}\\nLon: ${lon.toFixed(6)}\\n` +
                    `Precision: ±${acc.toFixed(0)} m`,
                    "ok"
                );

                try {
                    const resp = await fetch("/gps", {
                        method: "POST",
                        headers: { "Content-Type": "application/json" },
                        body: JSON.stringify({ latitude: lat, longitude: lon, accuracy: acc })
                    });
                    const data = await resp.json();

                    if (data.status === "cerrado") {
                        clearInterval(intervalo);
                        activo = false;
                        document.getElementById("btn").textContent = "Iniciar captura";
                        log("Sesion terminada por limite de tiempo.", "error");
                    }
                } catch(e) {
                    log("Error al enviar al servidor:\\n" + e.message, "error");
                }
            },
            function(error) {
                log(`Error GPS: ` + error.message, "error");
            },
            { enableHighAccuracy: true, timeout: 15000, maximumAge: 0 }
        );
    }
</script>
</body>
</html>
"""

inicio = time.time()
DURACION = 4 * 60 * 60

# Ruta que recibe los datos espaciales y los guarda
@app.route("/gps", methods=["POST"])
def gps():
    global inicio

    if time.time() - inicio > DURACION:
        return jsonify({"status": "cerrado"})

    data = request.json
    if not data or "latitude" not in data or "longitude" not in data:
        return jsonify({"status": "error", "msg": "Datos incompletos"}), 400

    feature = {
        "type": "Feature",
        "geometry": {
            "type": "Point",
            "coordinates": [data["longitude"], data["latitude"]]
        },
        "properties": {
            "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
            "accuracy": data.get("accuracy", None)
        }
    }

    with file_lock:
        geojson = load_geojson()
        geojson["features"].append(feature)
        with open(GEOJSON_FILE, "w") as f:
            json.dump(geojson, f, indent=2)

    return jsonify({"status": "ok"})

# Funcion para ejecutar Flask sin bloquear la libreta
def run_flask():
    app.run(host="0.0.0.0", port=5000, debug=False, use_reloader=False)

flask_thread = threading.Thread(target=run_flask)
flask_thread.daemon = True
flask_thread.start()
print("Servidor Flask corriendo en el puerto 5000 en segundo plano.")

Conectando al motor de ArcGIS...
Conexion establecida para Flask con exito.
Servidor Flask corriendo en el puerto 5000 en segundo plano.
 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://192.168.0.5:5000
Press CTRL+C to quit


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://192.168.0.5:5000
Press CTRL+C to quit


In [3]:
# Renderizado del mapa interactivo
mapa_monitoreo = gis_conexion.map("Monterrey, Nuevo Leon, Mexico")
mapa_monitoreo.basemap.basemap = 'streets-navigation-vector'
mapa_monitoreo

Map()

In [ ]:
# Bucle de actualizacion automatica
from IPython.display import clear_output
from arcgis.features import FeatureSet
import time

print("Iniciando monitoreo en tiempo real.")
print("Usa el boton Stop (cuadrado) en el menu superior para detener este bucle.")

puntos_dibujados = 0

try:
    while True:
        with open("gps_data.geojson", "r") as f:
            datos_actuales = json.load(f)
        
        total_puntos = len(datos_actuales.get("features", []))
        
        # Comprueba si Flask ha escrito puntos nuevos en el archivo
        if total_puntos > puntos_dibujados:
            
            # Prepara solo los puntos que aun no se han dibujado
            nuevos_puntos = {
                "type": "FeatureCollection",
                "features": datos_actuales["features"][puntos_dibujados:]
            }
            
            # Dibuja los puntos recientes en el widget de la celda superior
            if len(nuevos_puntos["features"]) > 0:
                fs = FeatureSet.from_geojson(nuevos_puntos)
                mapa_monitoreo.content.add(fs)
            
            # Limpia la consola y muestra el estatus actualizado
            clear_output(wait=True)
            puntos_dibujados = total_puntos
            print(f"Monitoreo activo. Puntos renderizados en el mapa: {puntos_dibujados}")
            
        # Espera 3 segundos antes de volver a leer el archivo
        time.sleep(3)
        
except KeyboardInterrupt:
    print("Monitoreo detenido correctamente por el usuario.")


Iniciando monitoreo en tiempo real.
Usa el boton Stop (cuadrado) en el menu superior para detener este bucle.
